### Obx events mapping as of 3/25/26
- 0 is analog visual
- 1 is analog audio #early 2026 sessions don't have this
- 2 is trial start
- 3 is frames
- 4 is left port
- 5 is center port
- 6 is right port

In [ ]:
from labdata.schema import (
    EphysRecording,
    SpikeSorting,
    Dataset,
    DatasetEvents,
    StreamSync,
)

from tqdm import tqdm

In [ ]:
# #for populating past nidq/obx events that haven't been populated already, run the following on hodgkin
# a = (EphysRecording).fetch("KEY")
# for k in a:
#     (EphysRecording & k).add_nidq_events()

### Populate the StreamSync table
Note: If a session doesn't have the DatasetEvents populated for both the imec and nidq/obx streams, this insertion will fail (foreign key constraint error).

In [ ]:
# posted here: https://gist.github.com/rojasgabriel/bdb29f3141a72d8c6237937541421ede


NIDQ_SUBJECTS = {
    "GRB006",
    "GRB036",
    "GRB041",
    "GRB045",
}  # assume all other mice were recorded with a obx

sorted_sessions = (EphysRecording & SpikeSorting).proj()
key = sorted_sessions.fetch("subject_name", "session_name", as_dict=True)
all_sessions = (Dataset() & key).proj()

for ses in tqdm(all_sessions):
    try:
        ephys_dset = (
            EphysRecording() & f'session_name = "{ses["session_name"]}"'
        ).fetch("dataset_name")[
            0
        ]  # get the specific "ephys_g*" since there can be multiple runs per day
    except Exception as e:
        print(f"WARNING: Couldn't find an ephys session for {ses}.\nException: {e}.")
        continue

    # determine clock system based on subject
    is_nidq = ses["subject_name"] in NIDQ_SUBJECTS
    clock_stream = "nidq" if is_nidq else "obx"

    if ses["dataset_name"] == "chipmunk":
        source_stream = "bpod"
        sync_entry = dict(
            ses,
            stream_name=source_stream,
            event_name="sync",
            clock_dataset=ephys_dset,
            clock_stream=clock_stream,
            clock_stream_event=2
            if is_nidq
            else "io2",  # sync/trial start input on board
        )
    else:
        source_stream = "imec0"
        sync_entry = dict(
            ses,
            stream_name=source_stream,
            event_name="6",
            clock_dataset=ephys_dset,
            clock_stream=clock_stream,
            clock_stream_event=0 if is_nidq else 6,  # hardware sync/trigger signal
        )

    # Filtering to sorted sessions doesn't guarantee DatasetEvents are populated
    # for the streams StreamSync references. Skip sessions where either the
    # source or the clock stream is missing rather than catching the resulting
    # foreign-key error after the fact.
    source_key = {**ses, "stream_name": source_stream}
    clock_key = {
        "subject_name": ses["subject_name"],
        "session_name": ses["session_name"],
        "dataset_name": ephys_dset,
        "stream_name": clock_stream,
    }
    if not (DatasetEvents() & source_key) or not (DatasetEvents() & clock_key):
        print(
            f"Skipped {ses['subject_name']}/{ses['session_name']}: missing "
            f"DatasetEvents for source={source_stream} or clock={clock_stream}"
        )
        continue

    StreamSync().insert([sync_entry], skip_duplicates=True)